# 63. The frozen-fold OOF library, thirty-four members

**One variable against ledger row 146** (`stack_prune35_public2`, CV 0.969406): the member set.

## Provenance, established by transitivity rather than by trust

`srcL/s6e8-oof-library-47-models` publishes 74 models and a README stating every one uses
`StratifiedKFold(n_splits=5, shuffle=True, random_state=42)` in original row order. A README is a
claim. The check is that **three of its members are models this repo already verified
independently**, through their own authors' printed per-fold AUCs, and two of those are
**bit-identical** to our verified copies: `pub_rmlp` and `pub_tabm` at Spearman 1.00000000 with a
maximum per-fold AUC difference of 0.00e+00. The library sits on our partition.

All 34 members loaded here reproduce the manifest's published OOF AUC to within 4.9e-06, asserted
in the loader rather than eyeballed.

**Two exclusions, both on evidence.** The five `srcB*` members are dropped: srcL sourced them
from srcB's published OOF *dataset*, and this repo established that the dataset copy and srcB's own
*kernel* output are different vectors at Spearman 0.9929. The kernel version verifies against
printed fold AUCs; the dataset version has none. `pub_rmlp` and `pub_tabm` are dropped as exact
duplicates of members row 145 already holds.

The README also independently confirms this repo's rejection of `lookup_srcA` in row 146. srcL
retrained the Lookup-Transformer himself because the original "uses 10 folds, so his published
out-of-fold predictions cannot be stacked against this 5-fold library without leaking". Our gate
reached the same verdict from the fold AUCs alone, without knowing that.

## The case against, written first

**The offset is shrinking and that is the number to watch, not CV.** Own models +0.001266, five
public +0.001259, ten public +0.001224. Several of these authors early-stop on the fold they are
predicting, so their out-of-fold is chosen knowing those rows' labels and is optimistic in a way
their test predictions are not. **Adding 34 more such members should shrink the offset again, and
CV will overstate the leaderboard gain.** Any projection here must use the realised offset.

**Saturation, twelve times measured.** srcL's own manifest records `rmlp_lat3` gaining +0.00015
solo and **exactly 0.000000** in his blend at correlation 0.9994 with its parent. The same law.
Thirty-four members drawn largely from four families this stack already holds will overlap heavily.

## The prediction

**+0.00030 to +0.00070 on CV, and less than that on the leaderboard.** `lookup` is the member to
watch: srcL measures its maximum correlation against every other member at 0.9869, where the
rest of the strong pack sits between 0.987 and 0.999, and it was worth +0.000109 to his blend,
more than any other single change he made.

Top 10 percent needs LB 0.97103. At the realised +0.001224 offset that is CV 0.969806, so this arm
needs about **+0.00040 CV** to reach it, and more if the offset shrinks again.


In [1]:
# 37_stack_views.ipynb
# Membership gate for the five vectors produced by notebooks 35 and 36.
# Runs locally: every member vector already lives in artifacts/oof.
import hashlib
import json
import pathlib
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

ROOT = next(b for b in [Path.cwd(), *Path.cwd().parents]
            if (b / "data" / "raw" / "train.csv").exists())
O = ROOT / "artifacts" / "oof"
S = ROOT / "submissions"

train = pd.read_csv(ROOT / "data" / "raw" / "train.csv")
test = pd.read_csv(ROOT / "data" / "raw" / "test.csv")
y = train["addicted_label"].to_numpy(np.int8)

# The fold vector is rebuilt rather than loaded, and then checked. A silently different
# fold vector is the one error here that produces a clean-looking wrong answer.
folds = np.full(len(train), -1, dtype=np.int64)
for i, (_, va) in enumerate(StratifiedKFold(5, shuffle=True,
                                            random_state=42).split(train, y)):
    folds[va] = i
assert (folds >= 0).all() and np.bincount(folds).sum() == len(train)
FOLD_SHA = hashlib.sha256(folds.tobytes()).hexdigest()[:16]
assert FOLD_SHA == "ec282b0968059676", FOLD_SHA
print(f"train {len(train):,}  test {len(test):,}  folds {np.bincount(folds)}")
print(f"fold sha {FOLD_SHA}  VERIFIED")

train 691,369  test 296,302  folds [138274 138274 138274 138274 138273]
fold sha ec282b0968059676  VERIFIED


In [2]:
# Row 94's forty-one minus the duplicate, in row 94's order, then the five candidates.
BASE = [
    ("te42", "te_bag42"), ("te2024", "te_seed2024"), ("te7", "te_seed7"),
    ("te2025", "te_seed2025"), ("te13", "te_seed13"),
    ("anchor", "lgbm_default_anchor_seed42"), ("trees300", "lgbm_trees300_seed42"),
    ("trees1000", "lgbm_trees1000_seed42"), ("trees2000", "lgbm_trees2000_seed42"),
    ("lr010", "lgbm_lr01_n1000_seed42"), ("lr005", "lgbm_lr005_n2000_seed42"),
    ("lr003", "lgbm_lr003_n3333_seed42"),
    ("bag42", "lgbm_bag08_lr005_n2000_seed42"),
    ("bag2024", "lgbm_bag08_lr005_n2000_seed2024"),
    ("bag7", "lgbm_bag08_lr005_n2000_seed7"),
    ("bag2025", "lgbm_bag08_lr005_n2000_seed2025"),
    ("bag13", "lgbm_bag08_lr005_n2000_seed13"),
    ("neural", "neural"), ("cat42", "catboost_te"), ("cat2024", "catboost_te_seed2024"),
    ("cat7", "catboost_te_seed7"), ("cat2025", "catboost_te_seed2025"),
    ("cat13", "catboost_te_seed13"), ("neural_te", "neural_te"),
    ("xgb_te", "xgb_te"), ("xgb2024", "xgb_te_seed2024"), ("xgb7", "xgb_te_seed7"),
    ("xgb2025", "xgb_te_seed2025"), ("xgb13", "xgb_te_seed13"),
    ("pair_top9", "xgb_pair_top9"),
]
BASE += [("cat_nat_c1", "cat_native_c1"), ("cat_nat_c2", "cat_native_c2"),
         ("xgb_raw", "xgb_raw"), ("cat_raw", "cat_raw"),
         ("xgb_te_fe", "xgb_te_fe"), ("cat_te_n4000", "cat_te_n4000"),
         ("xgb_raw_fe", "xgb_raw_fe"), ("cat_raw_n10k", "cat_raw_n10000"),
         ("lgb_raw_fe", "lgb_raw_fe"), ("cat_raw_fe", "cat_raw_fe")]
# lgb_raw is deliberately absent: it is bag42 re-run in another kernel, Pearson 0.999970 on
# logits. See the header and the 2026-08-22 entry in NOTES.md. Dropping it costs -0.000001.
DROPPED = [("lgb_raw", "duplicate of bag42, Pearson 0.999970")]

BASE += [("cat_te_fe", "cat_te_fe"), ("hgb_te_fe", "hgb_te_fe"),
         ("lgb_te_fe", "lgb_te_fe"), ("rf_te_fe", "rf_te_fe"),
         ("neural_lookup", "neural_lookup"), ("et_te_fe", "et_te_fe"),
         ("logit_te_fe", "logit_te_fe")]
BASE += [("xgb_tuned", "xgb_tuned")]
BASE += [("neural_fe", "neural_fe"), ("neural_wide", "neural_wide"),
         ("neural_res", "neural_res")]
BASE += [("realmlp", "realmlp")]
BASE += [("realmlp10", "realmlp10")]
BASE += [("realmlp_raw_fe", "realmlp_raw_fe")]
BASE += [("tabm", "tabm")]
# THE ONE VARIABLE. Row 144 held these fifty-five, all built by this repo. The five
# candidates below were not. Each is admitted only by writeup/verify_public_oof.py,
# which proves the fold partition matches ours by reproducing the author's own printed
# per-fold AUCs from our fold vector. See the header.
# Row 145's five, now part of the base.
PRIOR_PUBLIC = ["cb_srcE", "lgb_srcE", "xgb_srcA", "realmlp_srcA", "tabm_srcA",
                "lgb_srcB", "realmlp_srcI", "hgb_srcH", "xgb_srcC", "resnet_kava"]
PUBLIC = PRIOR_PUBLIC
# THE ONE VARIABLE. Thirty-four members from the frozen-fold library, loaded as raw
# .npy rather than through the kernel gate, because their provenance is established by
# transitivity against pub_rmlp and pub_tabm being bit-identical. See the header.
SZY = sorted((ROOT / "artifacts" / "wide_library" / "picked.txt").read_text().split())
CAND = [(n, n) for n in SZY]


def load(stem, kind):
    # OOF/test vector. Two naming conventions exist in artifacts/oof. The bare
    # "{stem}.npy" form is the OOF side only: the early LightGBM members never had a
    # test .npy written and their test side lives in submissions/. Falling back to the
    # bare name for kind="test" silently returns the OOF vector, which is caught by the
    # length assert below only because train and test differ in length.
    cands = [O / f"{stem}_{kind}.npy"]
    if kind == "oof":
        cands.append(O / f"{stem}.npy")
    for c in cands:
        if c.exists():
            return np.load(c)
    if kind == "test" and (S / f"{stem}.csv").exists():
        df = pd.read_csv(S / f"{stem}.csv")
        # A csv written in a different row order blends perfectly cleanly and is
        # undetectable in the score. Checked rather than assumed.
        assert (df["id"].to_numpy() == test["id"].to_numpy()).all(), f"id order {stem}"
        return df["addicted_label"].to_numpy()
    raise FileNotFoundError(f"{stem} {kind}")


# Row 145 already holds PRIOR_PUBLIC, so they belong in the index alongside our own.
MEM = BASE + [(n, n) for n in PRIOR_PUBLIC] + CAND
names = [n for n, _ in MEM]
Poof = {n: load(s, "oof") for n, s in BASE}
Ptest = {n: load(s, "test") for n, s in BASE}

# The public five, read from artifacts/public_oof/ with their id order asserted rather
# than assumed. A csv in a different row order blends perfectly cleanly and is invisible
# in the score, which is the failure row 59's loader already guards against.
PUB = ROOT / "artifacts" / "public_oof"
VER = {r["name"]: r for r in json.loads((PUB / "verification.json").read_text(encoding="utf-8"))}


def read_vec(path, n_expected, order_ref):
    """Mirror of writeup/verify_public_oof.load_vector, so a member is loaded here
    exactly as it was loaded when it was verified. Two shapes exist in the wild: a
    csv with or without an id column, and a bare .npy."""
    path = pathlib.Path(path)
    if path.suffix == ".npy":
        v = np.load(path)
        assert len(v) == n_expected, f"{path.name} has {len(v)} rows"
        return v.astype(float)
    df = pd.read_csv(path)
    assert len(df) == n_expected, f"{path.name} has {len(df)} rows"
    idc = [c for c in df.columns if c.lower() == "id"]
    if idc:
        # A csv in a different row order blends perfectly cleanly and is invisible in
        # the score, so this is asserted rather than hoped for.
        assert (df[idc[0]].to_numpy() == order_ref).all(), f"id order {path.name}"
    pref = [c for c in df.columns if any(k in c.lower() for k in ("oof", "pred", "prob"))]
    col = pref or [c for c in df.columns
                   if c.lower() not in ("id", "addicted_label", "target", "fold")]
    col = col or [c for c in df.columns if c.lower() != "id"]
    return df[col[0]].to_numpy(float)


for n in PUBLIC:
    r = VER.get(n)
    assert r and r["verdict"].startswith("ADMISSIBLE"), \
        f"{n} is not admissible: {r['verdict'] if r else 'absent'}. Re-run the gate."
    d = PUB / r["folder"]
    Poof[n] = read_vec(d / r["oof_file"], len(train), train["id"].to_numpy())
    Ptest[n] = read_vec(d / r["test_file"], len(test), test["id"].to_numpy())
# The library members. The manifest AUC is asserted, so a truncated or wrong file
# cannot enter quietly.
import csv as _csv
SZD = ROOT / "artifacts" / "wide_library"
_man = {r["model"]: float(r["oof_auc"])
        for r in _csv.DictReader((SZD / "manifest.csv").open(encoding="utf-8"))}
for n in SZY:
    o = np.load(SZD / f"oof_{n}.npy")
    t = np.load(SZD / f"test_{n}.npy")
    assert o.shape == (len(train),) and t.shape == (len(test),), n
    _a = roc_auc_score(y, o)
    assert abs(_a - _man[n]) < 5e-5, f"{n}: AUC {_a:.6f} vs manifest {_man[n]}"
    Poof[n], Ptest[n] = o.astype(float), t.astype(float)
print(f"{len(SZY)} library members loaded, all matching the manifest AUC to under 5e-5")
print(f"{len(PUBLIC)} kernel-verified public members loaded")
print(f"  kernel-verified     : {len(PRIOR_PUBLIC)}")
print(f"  library candidates  : {len(SZY)}")
_rej = [k for k, v in VER.items() if not v["verdict"].startswith("ADMISSIBLE")]
print(f"  rejected by the gate: {_rej}")

for n in names:
    assert Poof[n].shape == (len(train),), n
    assert Ptest[n].shape == (len(test),), n
    assert np.isfinite(Poof[n]).all() and np.isfinite(Ptest[n]).all(), n
    # A partially failed run leaves a constant fold, which blends silently.
    assert min(np.ptp(Poof[n][folds == f]) for f in range(5)) > 0, f"dead fold in {n}"

# Exact-duplicate quarantine. A duplicated array silently DOUBLES that model's weight.
# Row 59 found xgb_pair_base bit-identical to xgb_te this way and excluded it.
seen = {}
for n in names:
    h = hashlib.md5(np.ascontiguousarray(Poof[n]).tobytes()).hexdigest()
    assert h not in seen, f"{n} is bit-identical to {seen[h]}"
    seen[h] = n
print(f"{len(names)} vectors loaded, no exact duplicates")

# THE CORRELATION SCREEN. Hash equality cannot see one configuration run in two kernels:
# the arrays differ by thread-level numerical noise. bag42 and lgb_raw sat in row 94 at
# Pearson 0.999970 and no check fired. Three lines, and it would have caught them.
Lz = np.column_stack([np.clip(np.log(np.clip(Poof[n], 1e-9, 1 - 1e-9)
                                     / (1 - np.clip(Poof[n], 1e-9, 1 - 1e-9))), -30, 30)
                      for n in names])
Cm = np.corrcoef(((Lz - Lz.mean(0)) / Lz.std(0)).T)
np.fill_diagonal(Cm, 0.0)
near = [(names[i], names[j], Cm[i, j])
        for i in range(len(names)) for j in range(i + 1, len(names))
        if abs(Cm[i, j]) > 0.9999]
print(f"pairs above 0.9999: {len(near)}")
for a, b, r in near:
    print(f"  NEAR-DUPLICATE {a} and {b} at {r:+.6f}")
assert not near, "a near-duplicate pair is present, justify it or drop one"
print(f"removed from row 94's set: {[d[0] for d in DROPPED]}")
hi = sorted(((abs(Cm[i, j]), names[i], names[j])
             for i in range(len(names)) for j in range(i + 1, len(names))),
            reverse=True)[:3]
print("most collinear surviving pairs: "
      + ", ".join(f"{a}/{b} {r:.5f}" for r, a, b in hi))

34 library members loaded, all matching the manifest AUC to under 5e-5
10 kernel-verified public members loaded
  kernel-verified     : 10
  library candidates  : 34
  rejected by the gate: ['lookup_srcA', 'spline_srcD']


99 vectors loaded, no exact duplicates


pairs above 0.9999: 0
removed from row 94's set: ['lgb_raw']
most collinear surviving pairs: realmlp/realmlp10 0.99948, rmlp_lat/rmlp_lat3 0.99942, cat42/cat7 0.99907


In [3]:
def logit(p):
    p = np.clip(np.asarray(p, dtype=float), 1e-9, 1 - 1e-9)
    return np.clip(np.log(p / (1 - p)), -30, 30)


Loof = np.column_stack([logit(Poof[n]) for n in names])
Ltest = np.column_stack([logit(Ptest[n]) for n in names])
IDX = {n: i for i, n in enumerate(names)}
BASE65 = [IDX[n] for n, _ in BASE] + [IDX[n] for n in PRIOR_PUBLIC]

print("candidate solo CV, and disagreement with the members it most resembles:")
print(f"  {'candidate':12} {'solo CV':>10} {'rho vs xgb_te':>15} {'rho vs cat42':>14}")
for n, _ in CAND:
    cv = np.mean([roc_auc_score(y[folds == f], Poof[n][folds == f]) for f in range(5)])
    r1 = pd.Series(Poof[n]).corr(pd.Series(Poof["xgb_te"]), method="spearman")
    r2 = pd.Series(Poof[n]).corr(pd.Series(Poof["cat42"]), method="spearman")
    print(f"  {n:12} {cv:10.6f} {r1:15.6f} {r2:14.6f}")

# For scale: how decorrelated are two members that everyone agrees are near-copies?
r_seed = pd.Series(Poof["xgb_te"]).corr(pd.Series(Poof["xgb2024"]), method="spearman")
print(f"\n  for scale, xgb_te vs xgb_te_seed2024 (same model, different seed): {r_seed:.6f}")
print("  NOTES.md refuted reasoning from Spearman to blend value on 66 pairs at r=+0.143.")
print("  This table is context, not a prediction.")

candidate solo CV, and disagreement with the members it most resembles:
  candidate       solo CV   rho vs xgb_te   rho vs cat42


  digit_cat      0.966667        0.982331       0.985740


  digit_xgb      0.966321        0.985358       0.979202


  imp_cat        0.966919        0.981568       0.984493


  imp_lgbm       0.966319        0.984096       0.976761


...

  tabm_imp       0.968268        0.985069       0.980382


  tabm_seed3     0.968703        0.989628       0.983568


  tabm_wide      0.968469        0.989889       0.984239


  tabm_x12       0.968548        0.988884       0.983719



  for scale, xgb_te vs xgb_te_seed2024 (same model, different seed): 0.997344
  NOTES.md refuted reasoning from Spearman to blend value on 66 pairs at r=+0.143.
  This table is context, not a prediction.


In [4]:
def run(cols):
    # Fold-wise logistic combiner. No weight is ever fitted on a row it is scored on.
    oof = np.zeros(len(train))
    tst = np.zeros((5, len(test)))
    cf = np.zeros((5, len(cols)))
    nit = []
    for f in range(5):
        tr, va = folds != f, folds == f
        clf = LogisticRegression(C=1.0, max_iter=2000).fit(Loof[np.ix_(tr, cols)], y[tr])
        # A non-converged lbfgs fit reads HIGHER than the truth, so convergence is
        # asserted rather than hoped for. Added 2026-08-21 after the public review
        # flagged it; measured at 38 to 40 iterations, so it has never been close.
        nit.append(int(np.max(clf.n_iter_)))
        oof[va] = clf.decision_function(Loof[np.ix_(va, cols)])
        tst[f] = clf.decision_function(Ltest[:, cols])
        cf[f] = clf.coef_[0]
    assert max(nit) < 2000, f"combiner did not converge, {nit}"
    per = np.array([roc_auc_score(y[folds == f], oof[folds == f]) for f in range(5)])
    return per, tst, cf, max(nit)


# Thirty-four single-candidate arms would be 34 extra fits for a table nobody acts on.
# The decision here is about the SET, so the set is the arm, plus the one member
# srcL's own manifest singles out as the most decorrelated thing he found.
ARMS = {"65_row146": BASE65}
ARMS["66_lookup"] = BASE65 + [IDX["lookup"]]
ARMS["66_all34"] = BASE65 + [IDX[n] for n, _ in CAND]

res = {a: run(cols) for a, cols in ARMS.items()}

# The prune, recomputed inside a committed notebook rather than trusted from the audit's
# scratch script. Keeps the top k members of the WINNING arm by absolute coefficient.
_best_cols = ARMS["66_all34"]
_coef = res["66_all34"][2].mean(axis=0)
_order = np.argsort(-np.abs(_coef))
for _k in (25, 35, 50, 65):
    if _k < len(_best_cols):
        ARMS[f"prune{_k}"] = sorted(_best_cols[i] for i in _order[:_k])
        res[f"prune{_k}"] = run(ARMS[f"prune{_k}"])
per = {a: r[0] for a, r in res.items()}

# Row 54 compared its base arm against 0.968932, which is row 137's PRUNED value rather
# than the 53-member figure. That labelling slip is recorded in row 139's notes and is
# corrected here: the base arm below IS row 139's fifty-three, and 0.968932 is its CV.
ROW146_CV, ROW140_CV = 0.969406, 0.969406
repro = per["65_row146"].mean() - ROW146_CV
print(f"reproduction of row 146: {per['65_row146'].mean():.6f} vs {ROW146_CV:.6f}"
      f"  delta {repro:+.2e}   {'REPRODUCED' if abs(repro) < 1e-4 else 'FAILED'}")
assert abs(repro) < 1e-4, "base arm does not reproduce row 146, do not log this run"
print(f"combiner max n_iter across all arms: {max(r[3] for r in res.values())} of 2000\n")

hdr = " ".join(f"{'fold ' + str(i):>9}" for i in range(5))
print(f"{'arm':14} {hdr} {'mean':>10} {'sd':>9}")
for a in ARMS:
    print(f"{a:14} " + " ".join(f"{v:9.6f}" for v in per[a])
          + f" {per[a].mean():10.6f} {per[a].std():9.6f}")

reproduction of row 146: 0.969398 vs 0.969406  delta -8.34e-06   REPRODUCED
combiner max n_iter across all arms: 70 of 2000

arm               fold 0    fold 1    fold 2    fold 3    fold 4       mean        sd
65_row146       0.968745  0.969501  0.969502  0.970017  0.969223   0.969398  0.000415
66_lookup       0.968900  0.969619  0.969678  0.970166  0.969386   0.969550  0.000412
66_all34        0.969116  0.969817  0.969830  0.970319  0.969547   0.969726  0.000393
prune25         0.969072  0.969798  0.969847  0.970278  0.969522   0.969704  0.000398
prune35         0.969127  0.969816  0.969863  0.970323  0.969554   0.969736  0.000393
prune50         0.969127  0.969825  0.969861  0.970315  0.969555   0.969737  0.000390
prune65         0.969136  0.969833  0.969837  0.970317  0.969562   0.969737  0.000387


In [5]:
FLOOR_MEAN, FLOOR_FOLDS = 0.00005, 4
# THE LEAK TRIPWIRE. NOTES.md: a feature that jumps CV by an implausible amount is a
# leak until proven otherwise. A verified partition should behave like any other member
# set; anything above +0.002 here means the verification missed something.
LEAK_ALARM = 0.002
base_per = per["65_row146"]

print("Paired against row 146's sixty-five. The gate is >= +0.00005 mean AND >= 4/5 folds.\n")
print(f"{'arm':14} {'paired mean':>13} {'paired sd':>11} {'folds':>7} {'t(4)':>8}  gate")
gate = {}
for a in ARMS:
    if a == "65_row146":
        continue
    d = per[a] - base_per
    wins = int((d > 0).sum())
    sd = d.std(ddof=1)
    t = d.mean() / (sd / np.sqrt(5)) if sd > 0 else float("inf")
    fired = bool(d.mean() >= FLOOR_MEAN and wins >= FLOOR_FOLDS)
    gate[a] = fired
    print(f"{a:14} {d.mean():+13.6f} {sd:11.6f} {wins:5d}/5 {t:8.2f}"
          f"  {'FIRES' if fired else 'under floor'}")
    assert d.mean() < LEAK_ALARM, (
        f"{a} gains {d.mean():+.6f}, above the {LEAK_ALARM} alarm. Treat as a leak and "
        f"re-run writeup/verify_public_oof.py before believing this.")

Paired against row 146's sixty-five. The gate is >= +0.00005 mean AND >= 4/5 folds.

arm              paired mean   paired sd   folds     t(4)  gate
66_lookup          +0.000152    0.000022     5/5    15.71  FIRES
66_all34           +0.000328    0.000026     5/5    28.07  FIRES
prune25            +0.000306    0.000032     5/5    21.41  FIRES
prune35            +0.000339    0.000032     5/5    23.74  FIRES
prune50            +0.000339    0.000032     5/5    23.38  FIRES
prune65            +0.000339    0.000033     5/5    23.14  FIRES


In [6]:
# Coefficients of the best arm, which is where the result actually lives. Row 59's
# lesson: a model that is null on its own can still take a large weight, and where that
# weight comes FROM is the thing worth reading.
best = max((a for a in ARMS if a != "65_row146"), key=lambda a: per[a].mean())
print(f"best arm by CV: {best}   {per[best].mean():.6f}\n")

cols_b, cols_0 = ARMS[best], ARMS["65_row146"]
cb = res[best][2].mean(axis=0)
c0 = res["65_row146"][2].mean(axis=0)
base_map = {names[c]: c0[i] for i, c in enumerate(cols_0)}

rows = []
for i, c in enumerate(cols_b):
    n = names[c]
    was = base_map.get(n, float("nan"))
    rows.append({"member": n, "coef": cb[i], "was": was, "shift": cb[i] - was})
tab = pd.DataFrame(rows).sort_values("coef", ascending=False)
print(tab.to_string(index=False, float_format=lambda v: f"{v:+.4f}"))

new = set(n for n, _ in CAND) & set(tab.member)
gained = tab[tab.member.isin(new)]["coef"].sum()
lost = -tab[~tab.member.isin(new)]["shift"].sum()
print(f"\nnew members carry {gained:+.4f} in total")
print(f"the existing fifty-three give up {lost:+.4f} of weight between them")
if abs(gained) > 1e-12:
    print(f"substitution covers {100 * lost / gained:.0f} percent of the new weight")

best arm by CV: prune65   0.969737

        member    coef     was   shift
    cat_nat_c2 +0.2400 +0.2635 -0.0235
        lookup +0.2277     NaN     NaN
  realmlp_srcI +0.1794 +0.2597 -0.0802
  realmlp_srcA +0.1356 +0.2231 -0.0875
     latr1_xgb +0.1275     NaN     NaN
   tabm_deeper +0.1164     NaN     NaN
      lgb_srcB +0.0864 +0.2140 -0.1276
    pub_tabnet +0.0858     NaN     NaN
  cat_te_n4000 +0.0806 +0.0967 -0.0161
      tabm_imp +0.0669     NaN     NaN
   latwide_cat +0.0638     NaN     NaN
     tabm_wide +0.0609     NaN     NaN
      imp_lgbm +0.0609     NaN     NaN
imp_lgbm_tuned +0.0596     NaN     NaN
    tabm_seed3 +0.0556     NaN     NaN
    lattri_xgb +0.0460     NaN     NaN
      pub_ravi +0.0451     NaN     NaN
       imp_cat +0.0447     NaN     NaN
    xgb_srcC +0.0397 +0.0661 -0.0264
   lattri_lgbm +0.0386     NaN     NaN
     hgb_te_fe +0.0379 +0.0373 +0.0006
     cat_te_fe +0.0351 +0.0358 -0.0007
       imp_xgb +0.0333     NaN     NaN
      tabm_x12 +0.0316     NaN

In [7]:
# Three decisions, kept separate. Bundling them was the error corrected in row 59.
print("1. GATE")
for a, f in gate.items():
    print(f"     {a:14} {'FIRES' if f else 'under floor'}")

print("\n2. MEMBERSHIP")
print("   A sub-floor addition is still kept and logged as negligible: row 32 kept four")
print("   CatBoost seeds at +0.000014 and row 34 kept neural_te at +0.000043. Membership")
print("   follows the sign and the fold count, not the floor.")
keep = [a for a in ARMS if a != "65_row146"
        and (per[a] - base_per).mean() > 0
        and int(((per[a] - base_per) > 0).sum()) >= 4]
print(f"   arms positive and >= 4/5 folds: {keep if keep else 'none'}")
print(f"   carried forward: {best} at {per[best].mean():.6f}")

print("\n3. SUBMISSION")
SUB = S / "stack_library.csv"
# The floor the gate uses, applied to the submission decision too. Row 142 recorded
# that these had different thresholds and that a two-millionth difference wrote a csv.
if per[best].mean() > ROW140_CV + FLOOR_MEAN:
    p = res[best][1].mean(axis=0)
    sub = pd.DataFrame({"id": test["id"].to_numpy(),
                        "addicted_label": (np.argsort(np.argsort(p)) + 0.5) / len(p)})
    assert len(sub) == len(test) and np.isfinite(sub["addicted_label"]).all()
    sub.to_csv(SUB, index=False)
    print(f"   wrote {SUB.name}, {len(sub):,} rows,"
          f" {sub['addicted_label'].nunique():,} distinct")
    print("   AUC reads order only, so the rank transform changes nothing and keeps the")
    print("   file comparable with the earlier stack submissions.")
else:
    print(f"   no submission: best arm {per[best].mean():.6f} does not beat row 140's "
          f"{ROW140_CV:.6f}")

print(f"\nledger lines:\n  name    stack_{best}\n  cv_mean {per[best].mean():.6f}"
      f"\n  cv_std  {per[best].std():.6f}")

1. GATE
     66_lookup      FIRES
     66_all34       FIRES
     prune25        FIRES
     prune35        FIRES
     prune50        FIRES
     prune65        FIRES

2. MEMBERSHIP
   A sub-floor addition is still kept and logged as negligible: row 32 kept four
   CatBoost seeds at +0.000014 and row 34 kept neural_te at +0.000043. Membership
   follows the sign and the fold count, not the floor.
   arms positive and >= 4/5 folds: ['66_lookup', '66_all34', 'prune25', 'prune35', 'prune50', 'prune65']
   carried forward: prune65 at 0.969737

3. SUBMISSION


   wrote stack_library.csv, 296,302 rows, 296,302 distinct
   AUC reads order only, so the rank transform changes nothing and keeps the
   file comparable with the earlier stack submissions.

ledger lines:
  name    stack_prune65
  cv_mean 0.969737
  cv_std  0.000387
